# Luminos Health — TCOC Cost Forecast Model

This notebook builds an end-to-end 6-month medical cost forecasting workflow for Luminos Health using monthly historical claims features. It performs EDA and leakage checks, trains and evaluates baseline and LightGBM models with MLflow tracking, registers the best model in Unity Catalog, generates Jan–Jun 2026 forecasts, writes them to a Delta table, and loads them into Lakebase Postgres for the app.

**Assets**
* Features: `phanitha_test_catalog.tcoc_ml.cost_forecast_features`
* Model: `phanitha_test_catalog.tcoc_ml.cost_forecast_model` (served by endpoint `cost_forecast_model`)
* Output: `phanitha_test_catalog.tcoc_gold.tcoc_cost_forecast` (+ Lakebase `tcoc_gold.tcoc_cost_forecast`)

_Source: copied from `.conversations/Luminos Health - TCOC Cost Forecast Model` (run of 2026-09-16). One-off app.py scaffolding cells were omitted._

In [0]:
%pip install lightgbm psycopg2-binary

In [0]:
import math
import os
import tempfile
import warnings
from pathlib import Path

import mlflow
import numpy as np
import pandas as pd
import seaborn as sns
from databricks.sdk import WorkspaceClient
from lightgbm import LGBMRegressor
from matplotlib import pyplot as plt
from mlflow import MlflowClient
from mlflow.models import infer_signature
from psycopg2 import connect
from psycopg2.extras import execute_values
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
sns.set_theme(style="whitegrid")

FEATURE_TABLE = "phanitha_test_catalog.tcoc_ml.cost_forecast_features"
OUTPUT_TABLE = "phanitha_test_catalog.tcoc_gold.tcoc_cost_forecast"
REGISTERED_MODEL_NAME = "phanitha_test_catalog.tcoc_ml.cost_forecast_model"
EXPERIMENT_NAME = "/Users/phanitha.kommareddi@databricks.com/luminos-health-tcoc-cost-forecast"
SEED = 42
FORECAST_HORIZON = 6
TRAIN_END = pd.Timestamp("2025-09-01")
VALID_START = pd.Timestamp("2025-10-01")
VALID_END = pd.Timestamp("2025-12-01")

mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment(EXPERIMENT_NAME)

print(f"Spark version: {spark.version}")
print(f"Feature table: {FEATURE_TABLE}")
print(f"Output table: {OUTPUT_TABLE}")
print(f"Model name: {REGISTERED_MODEL_NAME}")

In [0]:
raw_df = spark.table(FEATURE_TABLE).toPandas()
raw_df["service_month"] = pd.to_datetime(raw_df["service_month"])
raw_df = raw_df.sort_values(["line_of_business", "plan_type", "member_state", "service_month"]).reset_index(drop=True)

print(f"Rows: {len(raw_df):,}")
print(f"Columns: {len(raw_df.columns)}")
print(f"Date range: {raw_df['service_month'].min().date()} -> {raw_df['service_month'].max().date()}")
print(f"LOBs: {raw_df['line_of_business'].nunique()}, plans: {raw_df['plan_type'].nunique()}, states: {raw_df['member_state'].nunique()}")

feature_types = pd.DataFrame({
    "feature": raw_df.columns,
    "dtype": [str(dtype) for dtype in raw_df.dtypes],
    "kind": [
        "timestamp" if pd.api.types.is_datetime64_any_dtype(raw_df[col])
        else "numeric float" if pd.api.types.is_float_dtype(raw_df[col])
        else "numeric integer" if pd.api.types.is_integer_dtype(raw_df[col])
        else "categorical"
        for col in raw_df.columns
    ],
})
display(feature_types)
display(raw_df.head(5))

lob_monthly = (
    raw_df.groupby(["service_month", "line_of_business"], as_index=False)[["total_paid", "pmpm"]]
    .sum()
    .sort_values(["service_month", "line_of_business"])
)
display(lob_monthly)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for lob, grp in lob_monthly.groupby("line_of_business"):
    axes[0].plot(grp["service_month"], grp["total_paid"], marker="o", linewidth=2, label=lob)
axes[0].set_title("Monthly Total Paid by LOB")
axes[0].set_xlabel("Service Month")
axes[0].set_ylabel("Total Paid")
axes[0].tick_params(axis="x", rotation=45)
axes[0].legend()

axes[1].hist(raw_df["total_paid"], bins=30, alpha=0.75, label="total_paid")
axes[1].set_title("Distribution of Total Paid")
axes[1].set_xlabel("Total Paid")
axes[1].set_ylabel("Frequency")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.scatterplot(data=raw_df, x="paid_lag_1", y="total_paid", hue="line_of_business", alpha=0.7, ax=axes[0])
axes[0].set_title("Total Paid vs Prior-Month Paid")
sns.boxplot(data=raw_df, x="line_of_business", y="pmpm", ax=axes[1])
axes[1].set_title("PMPM Distribution by LOB")
axes[1].tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()

In [0]:
current_month_only_features = [
    "claim_count",
    "unique_members",
    "total_billed",
    "total_units",
    "avg_billed_per_claim",
    "avg_paid_per_claim",
    "denial_rate_pct",
    "oon_pct",
    "er_rate_per_1000",
    "surgery_cost_share_pct",
    "reimbursement_rate_pct",
]

selected_feature_columns = [
    "line_of_business",
    "plan_type",
    "member_state",
    "month_num",
    "quarter_num",
    "year_num",
    "month_index",
    "paid_lag_1",
    "paid_lag_2",
    "paid_lag_3",
    "paid_lag_6",
    "pmpm_lag_1",
    "pmpm_lag_2",
    "pmpm_lag_3",
    "claims_lag_1",
    "claims_lag_2",
    "members_lag_1",
    "paid_rolling_3m_avg",
    "paid_rolling_6m_avg",
    "pmpm_rolling_3m_avg",
    "pmpm_rolling_6m_avg",
    "claims_rolling_3m_avg",
    "denial_rate_rolling_3m_avg",
    "paid_mom_change",
    "pmpm_mom_change",
    "paid_volatility_3m",
]

analysis_df = raw_df.copy()
columns_to_numeric = list({
    "total_paid",
    "pmpm",
    "claim_count",
    "unique_members",
    "denial_rate_pct",
    *current_month_only_features,
    *selected_feature_columns,
})
for col in columns_to_numeric:
    if col in analysis_df.columns and col not in {"line_of_business", "plan_type", "member_state"}:
        analysis_df[col] = pd.to_numeric(analysis_df[col], errors="coerce")

numeric_corr = analysis_df[["total_paid"] + [c for c in selected_feature_columns if c not in {"line_of_business", "plan_type", "member_state"}] + current_month_only_features].corr(numeric_only=True)["total_paid"].drop("total_paid").sort_values(key=np.abs, ascending=False)
correlation_df = numeric_corr.rename("corr_with_total_paid").reset_index().rename(columns={"index": "feature"})
display(correlation_df.head(20))

suspect_high_corr = correlation_df[correlation_df["corr_with_total_paid"].abs() >= 0.95].copy()
suspect_high_corr["reason"] = "|corr| >= 0.95 with total_paid"
leakage_df = pd.DataFrame({
    "feature": current_month_only_features,
    "reason": "current-month feature not honestly available at forecast time",
})
if not suspect_high_corr.empty:
    leakage_df = pd.concat([leakage_df, suspect_high_corr[["feature", "reason"]]], ignore_index=True).drop_duplicates()
display(leakage_df.sort_values("feature"))

categorical_features = ["line_of_business", "plan_type", "member_state"]
numeric_features = [col for col in selected_feature_columns if col not in categorical_features]

model_df = analysis_df[["service_month", "total_paid", "pmpm", "claim_count", "unique_members", "denial_rate_pct"] + selected_feature_columns].copy()
train_mask = model_df["service_month"] <= TRAIN_END
valid_mask = (model_df["service_month"] >= VALID_START) & (model_df["service_month"] <= VALID_END)

train_df = model_df.loc[train_mask].copy()
valid_df = model_df.loc[valid_mask].copy()

X_train = train_df[selected_feature_columns]
y_train = train_df["total_paid"].astype(float)
X_valid = valid_df[selected_feature_columns]
y_valid = valid_df["total_paid"].astype(float)

print(f"Train rows: {len(train_df):,} | months: {sorted(train_df['service_month'].dt.strftime('%Y-%m').unique().tolist())}")
print(f"Validation rows: {len(valid_df):,} | months: {sorted(valid_df['service_month'].dt.strftime('%Y-%m').unique().tolist())}")
print("Split justification: time-based backtest using Feb-Sep 2025 for training and Oct-Dec 2025 as the untouched holdout.")

corr_plot_cols = ["total_paid", "paid_lag_1", "paid_lag_3", "paid_rolling_3m_avg", "paid_rolling_6m_avg", "pmpm_lag_1", "claims_lag_1", "members_lag_1", "paid_volatility_3m"]
fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(analysis_df[corr_plot_cols].corr(numeric_only=True), annot=True, fmt=".2f", cmap="coolwarm", ax=ax)
ax.set_title("Correlation Heatmap for Forecast-Time Features")
plt.tight_layout()
plt.show()

In [0]:
def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def make_preprocessor(scale_numeric: bool) -> ColumnTransformer:
    num_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        num_steps.append(("scaler", StandardScaler()))
    return ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
            ("num", Pipeline(num_steps), numeric_features),
        ]
    )


models = {
    "lightgbm": TransformedTargetRegressor(
        regressor=Pipeline(
            steps=[
                ("preprocessor", make_preprocessor(scale_numeric=False)),
                (
                    "model",
                    LGBMRegressor(
                        random_state=SEED,
                        n_estimators=300,
                        learning_rate=0.05,
                        num_leaves=31,
                        subsample=0.9,
                        colsample_bytree=0.9,
                        reg_alpha=0.1,
                        reg_lambda=0.1,
                    ),
                ),
            ]
        ),
        func=np.log1p,
        inverse_func=np.expm1,
        check_inverse=False,
    ),
    "linear_regression": TransformedTargetRegressor(
        regressor=Pipeline(
            steps=[
                ("preprocessor", make_preprocessor(scale_numeric=True)),
                ("model", LinearRegression()),
            ]
        ),
        func=np.log1p,
        inverse_func=np.expm1,
        check_inverse=False,
    ),
}

training_results = []
model_artifacts = {}
validation_predictions = []

for model_name, model in models.items():
    with mlflow.start_run(run_name=f"tcoc_cost_forecast_{model_name}") as run:
        run_id = run.info.run_id
        mlflow.set_tags({
            "project": "luminos-health-tcoc",
            "model_family": model_name,
            "forecast_horizon_months": str(FORECAST_HORIZON),
            "train_period": "2025-02_to_2025-09",
            "valid_period": "2025-10_to_2025-12",
            "target": "total_paid",
        })
        mlflow.log_params({
            "train_rows": len(train_df),
            "valid_rows": len(valid_df),
            "feature_count": len(selected_feature_columns),
            "categorical_feature_count": len(categorical_features),
            "numeric_feature_count": len(numeric_features),
            "target_transform": "log1p",
            "seed": SEED,
        })

        model.fit(X_train, y_train)
        valid_pred = np.maximum(model.predict(X_valid), 0.0)

        metrics = {
            "mape": float(mean_absolute_percentage_error(y_valid, valid_pred)),
            "rmse": rmse(y_valid, valid_pred),
            "mae": float(mean_absolute_error(y_valid, valid_pred)),
        }
        mlflow.log_metrics(metrics)

        pred_df = valid_df[["service_month", "line_of_business", "plan_type", "member_state"]].copy()
        pred_df["actual_total_paid"] = y_valid.values
        pred_df["predicted_total_paid"] = valid_pred
        pred_df["model_name"] = model_name
        validation_predictions.append(pred_df)

        with tempfile.TemporaryDirectory() as tmpdir:
            pred_path = Path(tmpdir) / f"validation_predictions_{model_name}.csv"
            pred_df.to_csv(pred_path, index=False)
            mlflow.log_artifact(str(pred_path), artifact_path="validation")

            if model_name == "lightgbm":
                fitted_pipeline = model.regressor_
                feature_names = fitted_pipeline.named_steps["preprocessor"].get_feature_names_out()
                importances = fitted_pipeline.named_steps["model"].feature_importances_
                importance_df = pd.DataFrame({"feature": feature_names, "importance": importances}).sort_values("importance", ascending=False)
                imp_path = Path(tmpdir) / "lightgbm_feature_importance.csv"
                importance_df.to_csv(imp_path, index=False)
                mlflow.log_artifact(str(imp_path), artifact_path="artifacts")
                display(importance_df.head(15))

        signature_input = X_train.head(min(100, len(X_train))).copy()
        signature_output = model.predict(signature_input)
        signature = infer_signature(signature_input, signature_output)
        model_info = mlflow.sklearn.log_model(
            sk_model=model,
            name="model",
            signature=signature,
            input_example=X_train.head(min(3, len(X_train))).copy(),
        )

        model_artifacts[model_name] = {
            "run_id": run_id,
            "model": model,
            "metrics": metrics,
            "model_uri": model_info.model_uri,
        }
        training_results.append({"model_name": model_name, **metrics, "run_id": run_id, "model_uri": model_info.model_uri})

results_df = pd.DataFrame(training_results).sort_values(["mape", "rmse", "mae"]).reset_index(drop=True)
display(results_df)

best_model_name = results_df.iloc[0]["model_name"]
best_run_id = results_df.iloc[0]["run_id"]
best_model_uri = results_df.iloc[0]["model_uri"]
best_model = model_artifacts[best_model_name]["model"]
best_metrics = model_artifacts[best_model_name]["metrics"]
interval_width = 1.96 * best_metrics["rmse"]

print(f"Best model: {best_model_name}")
print(f"Best model URI: {best_model_uri}")
print(f"Validation metrics: {best_metrics}")

pyfunc_model = mlflow.pyfunc.load_model(best_model_uri)
if pyfunc_model.metadata.signature is None:
    raise ValueError("Logged model is missing a signature.")
if pyfunc_model.input_example is None:
    raise ValueError("Logged model is missing an input example.")
mlflow.models.predict(model_uri=best_model_uri, input_data=pyfunc_model.input_example)

registered_version = mlflow.register_model(
    model_uri=best_model_uri,
    name=REGISTERED_MODEL_NAME,
    await_registration_for=300,
)
model_version = registered_version.version
print(f"Registered model version: {model_version}")

In [0]:
segment_cols = ["line_of_business", "plan_type", "member_state"]
base_month = raw_df["service_month"].min()
forecast_months = pd.date_range(raw_df["service_month"].max() + pd.offsets.MonthBegin(1), periods=FORECAST_HORIZON, freq="MS")

state_df = model_df.copy().sort_values(segment_cols + ["service_month"]).reset_index(drop=True)
future_rows = []

for forecast_month in forecast_months:
    month_rows = []
    for segment_key, seg_hist in state_df.groupby(segment_cols, sort=False):
        seg_hist = seg_hist.sort_values("service_month").reset_index(drop=True)
        latest = seg_hist.iloc[-1]
        total_paid_history = seg_hist["total_paid"].astype(float).tolist()
        pmpm_history = seg_hist["pmpm"].astype(float).tolist()
        claim_history = seg_hist["claim_count"].astype(float).tolist()
        member_history = seg_hist["unique_members"].astype(float).tolist()
        denial_history = seg_hist["denial_rate_pct"].astype(float).tolist()

        def hist_value(values, lag):
            return values[-lag] if len(values) >= lag else np.nan

        future_row = {
            "line_of_business": segment_key[0],
            "plan_type": segment_key[1],
            "member_state": segment_key[2],
            "service_month": forecast_month,
            "month_num": forecast_month.month,
            "quarter_num": ((forecast_month.month - 1) // 3) + 1,
            "year_num": forecast_month.year,
            "month_index": (forecast_month.year - base_month.year) * 12 + (forecast_month.month - base_month.month) + 1,
            "paid_lag_1": hist_value(total_paid_history, 1),
            "paid_lag_2": hist_value(total_paid_history, 2),
            "paid_lag_3": hist_value(total_paid_history, 3),
            "paid_lag_6": hist_value(total_paid_history, 6),
            "pmpm_lag_1": hist_value(pmpm_history, 1),
            "pmpm_lag_2": hist_value(pmpm_history, 2),
            "pmpm_lag_3": hist_value(pmpm_history, 3),
            "claims_lag_1": hist_value(claim_history, 1),
            "claims_lag_2": hist_value(claim_history, 2),
            "members_lag_1": hist_value(member_history, 1),
            "paid_rolling_3m_avg": float(np.mean(total_paid_history[-3:])) if len(total_paid_history) >= 3 else np.nan,
            "paid_rolling_6m_avg": float(np.mean(total_paid_history[-6:])) if len(total_paid_history) >= 6 else np.nan,
            "pmpm_rolling_3m_avg": float(np.mean(pmpm_history[-3:])) if len(pmpm_history) >= 3 else np.nan,
            "pmpm_rolling_6m_avg": float(np.mean(pmpm_history[-6:])) if len(pmpm_history) >= 6 else np.nan,
            "claims_rolling_3m_avg": float(np.mean(claim_history[-3:])) if len(claim_history) >= 3 else np.nan,
            "denial_rate_rolling_3m_avg": float(np.mean(denial_history[-3:])) if len(denial_history) >= 3 else np.nan,
            "paid_mom_change": hist_value(total_paid_history, 1) - hist_value(total_paid_history, 2) if len(total_paid_history) >= 2 else np.nan,
            "pmpm_mom_change": hist_value(pmpm_history, 1) - hist_value(pmpm_history, 2) if len(pmpm_history) >= 2 else np.nan,
            "paid_volatility_3m": float(np.std(total_paid_history[-3:], ddof=1)) if len(total_paid_history) >= 3 else np.nan,
        }

        future_feature_frame = pd.DataFrame([future_row])[selected_feature_columns]
        predicted_paid = float(np.maximum(best_model.predict(future_feature_frame)[0], 0.0))
        projected_members = float(latest["unique_members"]) if float(latest["unique_members"]) > 0 else 1.0
        predicted_pmpm = predicted_paid / projected_members

        record = {
            "forecast_month": forecast_month,
            "line_of_business": segment_key[0],
            "plan_type": segment_key[1],
            "member_state": segment_key[2],
            "predicted_paid": predicted_paid,
            "predicted_pmpm": predicted_pmpm,
            "lower_bound": max(predicted_paid - interval_width, 0.0),
            "upper_bound": predicted_paid + interval_width,
            "model_version": str(model_version),
        }
        month_rows.append(record)

        state_append = future_row.copy()
        state_append.update({
            "total_paid": predicted_paid,
            "pmpm": predicted_pmpm,
            "claim_count": float(latest["claim_count"]),
            "unique_members": projected_members,
            "denial_rate_pct": float(latest["denial_rate_pct"]),
        })
        future_rows.append(record)
        state_df = pd.concat([state_df, pd.DataFrame([state_append])], ignore_index=True)

forecast_df = pd.DataFrame(future_rows).sort_values(["forecast_month", "line_of_business", "plan_type", "member_state"]).reset_index(drop=True)
display(forecast_df.head(10))

audit_df = (
    forecast_df.groupby("forecast_month", as_index=False)[["predicted_paid", "lower_bound", "upper_bound"]]
    .sum()
    .sort_values("forecast_month")
)
display(audit_df)

spark.createDataFrame(forecast_df).write.mode("overwrite").saveAsTable(OUTPUT_TABLE)
print(f"Wrote forecast table: {OUTPUT_TABLE}")

In [0]:
import importlib
import subprocess
import sys

from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
if not hasattr(w, "postgres"):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "databricks-sdk>=0.118.0"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    importlib.invalidate_caches()
    for module_name in [name for name in list(sys.modules) if name == "databricks" or name.startswith("databricks.")]:
        sys.modules.pop(module_name, None)
    from databricks.sdk import WorkspaceClient
    w = WorkspaceClient()

username = w.current_user.me().user_name
load_df = forecast_df.copy()
load_df["forecast_month"] = pd.to_datetime(load_df["forecast_month"]).dt.date

branches = {
    "production": "projects/tcoc-app/branches/production/endpoints/primary",
    "dev": "projects/tcoc-app/branches/dev/endpoints/primary",
}

for branch_name, endpoint_name in branches.items():
    ep = w.postgres.get_endpoint(name=endpoint_name)
    token = w.postgres.generate_database_credential(endpoint=ep.name).token
    conn = connect(
        host=ep.status.hosts.host,
        port=5432,
        dbname="databricks_postgres",
        user=username,
        password=token,
        sslmode="require",
    )
    conn.autocommit = True
    cur = conn.cursor()
    cur.execute("CREATE SCHEMA IF NOT EXISTS tcoc_gold")
    cur.execute(
        """
        CREATE TABLE IF NOT EXISTS tcoc_gold.tcoc_cost_forecast (
            forecast_month DATE,
            line_of_business TEXT,
            plan_type TEXT,
            member_state TEXT,
            predicted_paid DOUBLE PRECISION,
            predicted_pmpm DOUBLE PRECISION,
            lower_bound DOUBLE PRECISION,
            upper_bound DOUBLE PRECISION,
            model_version TEXT
        )
        """
    )
    cur.execute("TRUNCATE TABLE tcoc_gold.tcoc_cost_forecast")
    rows = [tuple(row) for row in load_df[[
        "forecast_month",
        "line_of_business",
        "plan_type",
        "member_state",
        "predicted_paid",
        "predicted_pmpm",
        "lower_bound",
        "upper_bound",
        "model_version",
    ]].itertuples(index=False, name=None)]
    execute_values(
        cur,
        """
        INSERT INTO tcoc_gold.tcoc_cost_forecast (
            forecast_month,
            line_of_business,
            plan_type,
            member_state,
            predicted_paid,
            predicted_pmpm,
            lower_bound,
            upper_bound,
            model_version
        ) VALUES %s
        """,
        rows,
        page_size=1000,
    )
    cur.execute("SELECT COUNT(*) FROM tcoc_gold.tcoc_cost_forecast")
    inserted_rows = cur.fetchone()[0]
    print(f"{branch_name}: loaded {inserted_rows:,} rows to tcoc_gold.tcoc_cost_forecast")
    conn.close()

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# Check supported workloads
workload_response = w.api_client.do("GET", "/api/2.0/serving-endpoints:workload-configs")
supported_workloads = {
    config["workload_type"]: [size["key"] for size in config.get("workload_sizes", [])]
    for config in workload_response.get("workload_configs", [])
}
for wt, sizes in supported_workloads.items():
    print(f"{wt}: {sizes}")

# Confirm the registered model
from mlflow import MlflowClient
import mlflow
mlflow.set_registry_uri("databricks-uc")
registry_client = MlflowClient(registry_uri="databricks-uc")
rv = registry_client.get_model_version(name="phanitha_test_catalog.tcoc_ml.cost_forecast_model", version="1")
print(f"\nRegistered model: {rv.name} v{rv.version}")
print(f"Status: {rv.status}")
print(f"Source: {rv.source}")

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.errors import NotFound
from databricks.sdk.service.serving import (
    EndpointCoreConfigInput,
    ServedEntityInput,
    ServingModelWorkloadType,
)

w = WorkspaceClient()

registered_model_name = "phanitha_test_catalog.tcoc_ml.cost_forecast_model"
model_version = "1"
endpoint_name = "cost_forecast_model"
workload_type = "CPU"
workload_size = "Small"
scale_to_zero_enabled = True

served_entity = ServedEntityInput(
    entity_name=registered_model_name,
    entity_version=model_version,
    workload_type=ServingModelWorkloadType(workload_type),
    workload_size=workload_size,
    scale_to_zero_enabled=scale_to_zero_enabled,
)
endpoint_config = EndpointCoreConfigInput(served_entities=[served_entity])

try:
    existing_endpoint = w.serving_endpoints.get(endpoint_name)
except NotFound:
    existing_endpoint = None

if existing_endpoint is not None:
    current_entities = list(
        existing_endpoint.config.served_entities
        if existing_endpoint.config and existing_endpoint.config.served_entities
        else []
    )
    current_entity_names = [entity.entity_name for entity in current_entities]
    if len(current_entities) != 1 or current_entity_names[0] != registered_model_name:
        raise ValueError(
            f"Refusing to replace endpoint {endpoint_name!r}, which currently serves "
            f"{current_entity_names}. Choose a different name or confirm full replacement."
        )
    print(f"Updating existing endpoint '{endpoint_name}'...")
    w.serving_endpoints.update_config(
        name=endpoint_name,
        served_entities=[served_entity],
    )
else:
    print(f"Creating new endpoint '{endpoint_name}'...")
    w.serving_endpoints.create(
        name=endpoint_name,
        config=endpoint_config,
    )

print(f"Endpoint '{endpoint_name}' deployment initiated.")
print(f"  Model: {registered_model_name} v{model_version}")
print(f"  Workload: {workload_type} / {workload_size}")
print(f"  Scale to zero: {scale_to_zero_enabled}")
print("Deployment typically takes ~15 minutes. Polling for readiness...")

In [0]:
import time
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import EndpointStateConfigUpdate, EndpointStateReady

w = WorkspaceClient()
endpoint_name = "cost_forecast_model"

def wait_for_endpoint_ready(name, timeout_s=900, poll_s=15):
    deadline = time.time() + timeout_s
    failure_states = {
        EndpointStateConfigUpdate.UPDATE_FAILED,
        EndpointStateConfigUpdate.UPDATE_CANCELED,
    }
    while time.time() < deadline:
        state = w.serving_endpoints.get(name).state
        if (
            state.ready == EndpointStateReady.READY
            and state.config_update == EndpointStateConfigUpdate.NOT_UPDATING
        ):
            print(f"\n✓ Endpoint '{name}' is READY and serving.")
            ep = w.serving_endpoints.get(name)
            print(f"  URL: https://{w.config.host}/serving-endpoints/{name}/invocations")
            return
        if state.config_update in failure_states:
            raise RuntimeError(
                f"{name} deployment failed: {state.config_update.value}"
            )
        elapsed = int(time.time() - (deadline - timeout_s))
        print(f"  [{elapsed}s] State: ready={state.ready.value}, config_update={state.config_update.value}", end="\r")
        time.sleep(poll_s)
    raise TimeoutError(f"{name} not ready after {timeout_s}s")

wait_for_endpoint_ready(endpoint_name)